# Step 3: siamese wrapper
One `Encoder` (from `08_encoder_step.py`) called on **both** sides of a pair, then a symmetric comparison head → one logit per pair. The class lives in `08_encoder_step.py`; this notebook tests it.

| Piece | Spec |
|---|---|
| Encoder | **one** instance, `self.encoder`, called twice (weight sharing = the siamese property) |
| Comparison features | `torch.cat([(u - v).abs(), u * v], dim=-1)`: both order-free, so `score(a, b) == score(b, a)` |
| Head | `nn.Linear(2 * enc_dim, 1)` → `.squeeze(-1)` → logits `(B,)`; no sigmoid here (`BCEWithLogitsLoss` applies it) |
| Shapes | `ids_a/b (B, T)`, `len_a/b (B,)` → `u, v (B, 256)` → features `(B, 512)` → logits `(B,)` |

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import torch
from torch import nn

from helpers.pairs import load_pairs

_vocab_step = import_module("siamese_model.01_vocab_step")       # digit-prefixed filenames -> can't use `from ... import`
build_vocab = _vocab_step.build_vocab
PAD_ID, UNK_ID = _vocab_step.PAD_ID, _vocab_step.UNK_ID
encode_batch = import_module("siamese_model.03_encode_batch_step").encode_batch
Encoder = import_module("siamese_model.08_encoder_step").Encoder

In [2]:
ot_train, ot_train_label = load_pairs("overture", "train")   # only the train split: the vocab is built from train
vocab = build_vocab(ot_train)

## SiameseMatcher

In [3]:
# the class lives in 08_encoder_step.py (single copy, shared with training); this notebook tests it
SiameseMatcher = import_module("siamese_model.08_encoder_step").SiameseMatcher

## Tests
| # | Test | Expect |
|---|---|---|
| 1 | Shape | two pairs → logits `(2,)` |
| 2 | **Weight sharing** | total parameters = 167,168 + 513 (one encoder, not two); names start with `encoder.` / `head.` |
| 3 | **Swap test** | `eval()` + `no_grad()`: `allclose(model(a, b), model(b, a))`, with A and B of *different* lengths |
| 4 | Gradient reaches both sides | after `model(...).sum().backward()`, the embedding rows of characters that appear **only** in side B get a nonzero gradient |

In [4]:
a = ["aaa"]
b = ["bbb"]
x_a, len_a = encode_batch(a, vocab)
x_b, len_b = encode_batch(b, vocab)

encoder = Encoder(len(vocab))
matcher = SiameseMatcher(encoder)
params =  dict(matcher.named_parameters())

find_match = matcher(x_a, len_a, x_b, len_b)
find_match.backward()
#Test if different embeddings are updated when we commpare different characters
grad = params["encoder.embedder.emb.weight"].grad
assert grad[vocab['b']].abs().sum() > 0
assert (grad.abs().sum(1) != 0).sum() > 1
assert find_match.shape == (1,)

a = ["aaaggg", "bbb"]
b = ["ccc", "ddd"]
x_a, len_a = encode_batch(a, vocab)
x_b, len_b = encode_batch(b, vocab)

encoder = Encoder(len(vocab))
matcher = SiameseMatcher(encoder).eval()

#Check the output shape updates according to number of comparisons
with torch.no_grad():
    find_match = matcher(x_a, len_a, x_b, len_b)
assert find_match.shape == (2,)

# weight sharing: one encoder (167,168) + head (513); two encoders would give 2 x 167,168 + 513
assert sum(torch.numel(p) for name, p in matcher.named_parameters()) == 167168 + 513

#Verify that swapping comparison word sequence lists, does not affect the comparison output
with torch.no_grad():
    find_match_opp = matcher(x_b, len_b, x_a, len_a)
    assert torch.allclose(find_match, find_match_opp, atol=1e-6)
